# Tutorial 06 — Business Intelligence

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/06_business.ipynb)

**Forecasting is only the first step.** Real-world decision-making requires anomaly detection, what-if scenarios for planning, backtesting to validate your approach, and business-specific accuracy metrics.

| What you'll learn | Time |
|:------------------|:-----|
| Anomaly detection | 2 min |
| What-if scenarios | 2 min |
| Backtesting | 2 min |
| Business metrics | 2 min |
| Combined workflow | 2 min |

In [ ]:
!pip install -q vectrix

## 1. Anomaly Detection

Identify unusual observations in your historical data before forecasting.

In [ ]:
from vectrix.business import AnomalyDetector
import numpy as np

np.random.seed(42)
data = np.random.randn(200) * 10 + 100
data[50] = 200
data[120] = 30
data[175] = 250

detector = AnomalyDetector()
result = detector.detect(data, method="auto")

print(f"Method used:     {result.method}")
print(f"Anomalies found: {result.nAnomalies}")
print(f"Anomaly ratio:   {result.anomalyRatio:.1%}")
print(f"Anomaly indices: {result.indices}")

In [ ]:
# Compare detection methods
for method in ["zscore", "iqr", "rolling"]:
    r = detector.detect(data, method=method)
    print(f"{method:>8s}: {r.nAnomalies} anomalies at {r.indices}")

### Detection Methods

| Method | How It Works | Best For |
|--------|-------------|----------|
| `auto` | Automatically selects best method | General use (recommended) |
| `zscore` | Flags points > 3 std from mean | Normally distributed data |
| `iqr` | Flags points outside 1.5x IQR | Skewed distributions |
| `rolling` | Flags points outside rolling window | Non-stationary data |

## 2. What-If Analysis

Explore hypothetical scenarios against your baseline forecast — essential for budget planning and risk assessment.

In [ ]:
from vectrix.business import WhatIfAnalyzer
from vectrix import forecast
import numpy as np

np.random.seed(42)
data = np.random.randn(200).cumsum() + 500
result = forecast(data, steps=30)

analyzer = WhatIfAnalyzer()
scenarios = analyzer.analyze(
    result.predictions,
    data,
    [
        {"name": "Optimistic", "trend_change": 0.1},
        {"name": "Pessimistic", "trend_change": -0.15},
        {"name": "Supply Shock", "shock_at": 10, "shock_magnitude": -0.3, "shock_duration": 5},
        {"name": "Level Shift", "level_shift": 0.05},
    ]
)

for sr in scenarios:
    print(f"{sr.name:>15s}: mean={sr.predictions.mean():.2f}, impact={sr.impact:+.1f}%")

### Scenario Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `name` | `str` | Scenario label |
| `trend_change` | `float` | Percentage trend adjustment (0.1 = +10%) |
| `shock_at` | `int` | Step index where shock begins |
| `shock_magnitude` | `float` | Shock size (-0.3 = -30% drop) |
| `shock_duration` | `int` | Number of steps the shock lasts |
| `level_shift` | `float` | Permanent level adjustment (0.05 = +5%) |

## 3. Backtesting

Walk-forward validation: repeatedly train on past data and predict the next window.

In [ ]:
from vectrix.business import Backtester
from vectrix.engine.ets import AutoETS
import numpy as np

np.random.seed(42)
data = np.random.randn(300).cumsum() + 200

bt = Backtester(nFolds=5, horizon=14, strategy='expanding')
result = bt.run(data, lambda: AutoETS())

print(f"Average MAPE: {result.avgMAPE:.2f}%")
print(f"Average RMSE: {result.avgRMSE:.2f}")
print(f"Best fold:    #{result.bestFold}")
print(f"Worst fold:   #{result.worstFold}")

In [ ]:
# Per-fold breakdown
print("\nPer-fold breakdown:")
for f in result.folds:
    print(f"  Fold {f.fold}: MAPE={f.mape:.2f}%, RMSE={f.rmse:.2f}")

### Strategy Comparison

**Expanding window** — Each fold uses all data up to the cutoff. Recommended for most cases.
```
Fold 1: [====TRAIN====][TEST]
Fold 2: [======TRAIN======][TEST]
Fold 3: [========TRAIN========][TEST]
```

**Sliding window** — Fixed-size training window. Useful when older data is no longer relevant.
```
Fold 1: [====TRAIN====][TEST]
Fold 2:    [====TRAIN====][TEST]
Fold 3:       [====TRAIN====][TEST]
```

## 4. Business Metrics

MAPE tells you about statistical accuracy. Businesses care about different things: systematic bias, volume-weighted error, and naive baseline comparison.

In [ ]:
from vectrix.business import BusinessMetrics
import numpy as np

actual = np.array([100, 120, 110, 130, 140, 125, 135, 150, 145, 155])
predicted = np.array([105, 115, 112, 128, 145, 120, 138, 148, 140, 160])

metrics = BusinessMetrics()
result = metrics.calculate(actual, predicted)

print(f"Bias:                {result['bias']:+.2f}")
print(f"Bias %:              {result['biasPercent']:+.2f}%")
print(f"WAPE:                {result['wape']:.2f}%")
print(f"MASE:                {result['mase']:.2f}")
print(f"Accuracy:            {result['forecastAccuracy']:.1f}%")
print(f"Over-forecast ratio: {result['overForecastRatio']:.1%}")
print(f"Under-forecast:      {result['underForecastRatio']:.1%}")

### Interpreting MASE

MASE (Mean Absolute Scaled Error) compares your model to a Naive baseline:

- **MASE < 1.0** — Your model beats Naive. Good.
- **MASE = 1.0** — Your model equals Naive. No value added.
- **MASE > 1.0** — Naive would have been better. Investigate.

> **Tip:** WAPE is preferred over MAPE in business contexts because it handles near-zero values gracefully and weights errors by volume.

## 5. Combined Workflow

In practice, these tools work together: detect anomalies, backtest, forecast, then explore scenarios.

In [ ]:
import numpy as np
from vectrix import forecast
from vectrix.business import AnomalyDetector, Backtester, BusinessMetrics, WhatIfAnalyzer
from vectrix.engine.ets import AutoETS

np.random.seed(42)
data = np.random.randn(365).cumsum() + 1000

# Step 1: Anomaly check
detector = AnomalyDetector()
anomalies = detector.detect(data, method="auto")
print(f"1. Anomalies in history: {anomalies.nAnomalies}")

# Step 2: Backtest
bt = Backtester(nFolds=4, horizon=30, strategy='expanding')
bt_result = bt.run(data, lambda: AutoETS())
print(f"2. Backtest MAPE: {bt_result.avgMAPE:.2f}%")

# Step 3: Forecast
result = forecast(data, steps=30)
print(f"3. Model: {result.model}")

# Step 4: Scenarios
analyzer = WhatIfAnalyzer()
scenarios = analyzer.analyze(result.predictions, data, [
    {"name": "Growth +10%", "trend_change": 0.10},
    {"name": "Decline -10%", "trend_change": -0.10},
])
for s in scenarios:
    print(f"4. {s.name}: mean={s.predictions.mean():.0f}")

## 6. Monthly Sales Review

A realistic example — evaluating last month's forecast accuracy using business metrics.

In [ ]:
import numpy as np
from vectrix.business import BusinessMetrics

actual_last_month = np.array([
    320, 345, 310, 380, 400, 420, 350,
    330, 360, 325, 390, 410, 430, 365,
    340, 370, 335, 395, 415, 440, 375,
    345, 375, 340, 400, 425, 445, 380,
    350, 385
])

predicted_last_month = np.array([
    315, 340, 320, 370, 395, 415, 345,
    325, 355, 330, 385, 405, 425, 360,
    335, 365, 340, 390, 410, 435, 370,
    340, 370, 345, 395, 420, 440, 375,
    345, 380
])

metrics = BusinessMetrics()
result = metrics.calculate(actual_last_month, predicted_last_month)

print("=== Monthly Performance Review ===")
print(f"Forecast Accuracy: {result['forecastAccuracy']:.1f}%")
print(f"Bias:              {result['bias']:+.1f} units/day")
print(f"WAPE:              {result['wape']:.1f}%")
print(f"MASE:              {result['mase']:.2f}")

if result['mase'] < 1.0:
    print("\nModel outperforms Naive baseline.")
if abs(result['biasPercent']) > 5:
    direction = 'over' if result['bias'] > 0 else 'under'
    print(f"Warning: Systematic {direction}-forecasting detected.")

---

**Congratulations!** You've completed all 6 Vectrix tutorials.

**Resources:**
- [Full Documentation](https://eddmpython.github.io/vectrix/docs/)
- [GitHub](https://github.com/eddmpython/vectrix)
- [PyPI](https://pypi.org/project/vectrix/)